# 2D Rock-Paper-Scissors Territorial Walkers

Computational physics stochastic simulation project notebook.

## Introduction

This notebook simulates a 2D square lattice where walkers of three species (Rock, Paper, Scissors) move randomly and convert territory based on Rock-Paper-Scissors rules.

Grid state encoding:
- 0 = empty
- 1 = rock
- 2 = paper
- 3 = scissors

## Model and Rules

Each time step: every walker takes exactly one nearest-neighbor step (up/down/left/right), with periodic boundaries by default.

If a walker enters a destination cell:
1. Empty: destination becomes walker species.
2. Same species: no conflict.
3. Different species: apply RPS rule, convert both old and destination cells to winner, and update walker species to winner.

## Algorithm Plan

Pseudocode:

```text
initialize_grid_and_walkers(N, walkers_each_species, seed):
    grid <- zeros(N,N)
    create walkers with species [1,1,2,2,3,3] for debug case
    place on distinct random cells
    set grid at those cells to species

winner(a,b):
    return RPS winner

step_walker(walker, grid):
    choose random move from four directions
    apply periodic wrap
    inspect destination and update by rules

step_system(grid, walkers):
    process walkers in random order

run_simulation(...):
    iterate time steps
    store species fractions and optional snapshots
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

EMPTY, ROCK, PAPER, SCISSORS = 0, 1, 2, 3

STATE_COLORS = {
    EMPTY: "#f0f0f0",
    ROCK: "#000000",
    PAPER: "#0095DA",
    SCISSORS: "#e5ff23"
}

STATE_NAMES = {
    EMPTY: "Empty",
    ROCK: "Rock",
    PAPER: "Paper",
    SCISSORS: "Scissors"
}

cmap = ListedColormap([STATE_COLORS[s] for s in [EMPTY, ROCK, PAPER, SCISSORS]])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

## Implementation

In [ ]:
BASE_UPSET_CAPTURE_PROB = 0.025 # 2.5% per net local walker advantage against the RPS flow

def initialize_grid_and_walkers(n=10, walkers_each_species=2, seed=0):
    rng = np.random.default_rng(seed)
    grid = np.zeros((n, n), dtype=np.int8)

    species_list = ([ROCK] * walkers_each_species +
                    [PAPER] * walkers_each_species +
                    [SCISSORS] * walkers_each_species)

    total_walkers = len(species_list)
    if total_walkers > n * n:
        raise ValueError("Too many walkers for grid size.")

    flat_positions = rng.choice(n * n, size=total_walkers, replace=False)
    walkers = []
    for sp, pos in zip(species_list, flat_positions):
        x, y = divmod(int(pos), n)
        walkers.append({"x": x, "y": y, "species": int(sp)})
        grid[x, y] = sp

    return grid, walkers, rng


def winner(a, b):
    if a == b:
        return a
    if {a, b} == {ROCK, SCISSORS}:
        return ROCK
    if {a, b} == {SCISSORS, PAPER}:
        return SCISSORS
    if {a, b} == {PAPER, ROCK}:
        return PAPER
    raise ValueError(f"Invalid species pair: {a}, {b}")


def local_species_counts(walkers, n, m):
    """
    Build a per-cell species count tensor for current walker positions.
    shape = (n, m, 4), where last axis indexes [EMPTY, ROCK, PAPER, SCISSORS].
    """
    counts = np.zeros((n, m, 4), dtype=np.int32)
    for w in walkers:
        counts[w["x"], w["y"], w["species"]] += 1
    return counts


def against_flow_capture_probability(attacker_count, defender_count, base_prob=BASE_UPSET_CAPTURE_PROB):
    """
    Upset probability when attacker is moving against normal RPS flow.
    Example: attacker_count=4, defender_count=1 -> (4-1)*0.025 = 0.075 (7.5%).
    """
    advantage = max(0, int(attacker_count) - int(defender_count))
    return min(1.0, base_prob * advantage)


def step_walker(walker, grid, rng, cell_counts, periodic=True):
    n, m = grid.shape
    x_old, y_old = walker["x"], walker["y"]
    species = int(walker["species"])

    # Walker leaves old cell first (for local counting during destination encounter)
    cell_counts[x_old, y_old, species] = max(0, cell_counts[x_old, y_old, species] - 1)

    moves = np.array([(-1, 0), (1, 0), (0, -1), (0, 1)])
    dx, dy = moves[rng.integers(0, 4)]

    x_new, y_new = x_old + dx, y_old + dy
    if periodic:
        x_new %= n
        y_new %= m
    else:
        x_new = min(max(x_new, 0), n - 1)
        y_new = min(max(y_new, 0), m - 1)

    destination = int(grid[x_new, y_new])
    final_species = species

    if destination == EMPTY:
        # Empty destination: walker occupies it
        grid[x_new, y_new] = species

    elif destination == species:
        # Same species destination: no conflict
        pass

    else:
        # Different species: weighted probabilistic conflict
        normal_winner = winner(species, destination)

        # Counts inside destination cell, including the arriving attacker
        attacker_count = int(cell_counts[x_new, y_new, species]) + 1
        defender_count = int(cell_counts[x_new, y_new, destination])

        if normal_winner == species:
            # With the flow: attacker wins deterministically
            conflict_winner = species
        else:
            # Against the flow: attacker can upset with weighted probability
            p_upset = against_flow_capture_probability(attacker_count, defender_count)
            conflict_winner = species if rng.random() < p_upset else destination

        grid[x_old, y_old] = conflict_winner
        grid[x_new, y_new] = conflict_winner
        final_species = conflict_winner

    walker["x"], walker["y"], walker["species"] = x_new, y_new, int(final_species)
    cell_counts[x_new, y_new, int(final_species)] += 1


def step_system(grid, walkers, rng, periodic=True):
    n, m = grid.shape
    cell_counts = local_species_counts(walkers, n, m)
    order = rng.permutation(len(walkers))
    for i in order:
        step_walker(walkers[i], grid, rng, cell_counts, periodic=periodic)


def compute_species_fractions(grid):
    counts = np.bincount(grid.ravel(), minlength=4)
    return counts / grid.size


def run_simulation(n=50, walkers_each_species=10, steps=600, seed=0, snapshot_times=None, periodic=True):
    grid, walkers, rng = initialize_grid_and_walkers(n=n, walkers_each_species=walkers_each_species, seed=seed)

    history = np.zeros((steps + 1, 4), dtype=float)
    history[0] = compute_species_fractions(grid)

    if snapshot_times is None:
        snapshot_times = [0, steps // 4, steps // 2, steps]
    snapshot_times = sorted(set(snapshot_times))

    snapshots = {}
    if 0 in snapshot_times:
        snapshots[0] = grid.copy()

    for t in range(1, steps + 1):
        step_system(grid, walkers, rng, periodic=periodic)
        history[t] = compute_species_fractions(grid)
        if t in snapshot_times:
            snapshots[t] = grid.copy()

    return grid, walkers, history, snapshots

In [ ]:
def plot_grid(grid, ax=None, title="Grid state"):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(grid, cmap=cmap, norm=norm, origin="lower", interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("y")
    ax.set_ylabel("x")

    legend_items = [
        Patch(facecolor=STATE_COLORS[s], edgecolor="k", label=f"{s}: {STATE_NAMES[s]}")
        for s in [EMPTY, ROCK, PAPER, SCISSORS]
    ]
    ax.legend(handles=legend_items, bbox_to_anchor=(1.02, 1), loc="upper left")


def plot_species_history(history, ax=None, title="Species fractions vs time"):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))

    t = np.arange(history.shape[0])
    ax.plot(t, history[:, ROCK], label="Rock", color=STATE_COLORS[ROCK], lw=2)
    ax.plot(t, history[:, PAPER], label="Paper", color=STATE_COLORS[PAPER], lw=2)
    ax.plot(t, history[:, SCISSORS], label="Scissors", color=STATE_COLORS[SCISSORS], lw=2)
    ax.plot(t, history[:, EMPTY], label="Empty", color=STATE_COLORS[EMPTY], lw=1.5, ls="--")
    ax.set_xlabel("Time step")
    ax.set_ylabel("Fraction of cells")
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend()

## Verification Tests

The tests below verify initial setup, RPS winner logic, grid-state validity, and periodic-boundary walker consistency.

In [ ]:
def run_verification_tests(verbose=True):
    # 1) Initial-condition conservation
    n = 10
    k = 2
    grid0, walkers0, _ = initialize_grid_and_walkers(n=n, walkers_each_species=k, seed=123)
    assert len(walkers0) == 3 * k

    wc = {ROCK: 0, PAPER: 0, SCISSORS: 0}
    for w in walkers0:
        wc[w["species"]] += 1
    assert wc[ROCK] == k and wc[PAPER] == k and wc[SCISSORS] == k

    counts0 = np.bincount(grid0.ravel(), minlength=4)
    assert counts0[ROCK] == k and counts0[PAPER] == k and counts0[SCISSORS] == k

    # 2) Winner-function correctness
    assert winner(ROCK, SCISSORS) == ROCK and winner(SCISSORS, ROCK) == ROCK
    assert winner(SCISSORS, PAPER) == SCISSORS and winner(PAPER, SCISSORS) == SCISSORS
    assert winner(PAPER, ROCK) == PAPER and winner(ROCK, PAPER) == PAPER

    # 3 + 4) Grid and walker consistency after evolution
    grid_t, walkers_t, _, _ = run_simulation(n=30, walkers_each_species=4, steps=300, seed=7)
    assert set(np.unique(grid_t)).issubset({EMPTY, ROCK, PAPER, SCISSORS})

    n2 = grid_t.shape[0]
    for w in walkers_t:
        assert 0 <= w["x"] < n2 and 0 <= w["y"] < n2

    if verbose:
        print("All verification tests passed.")

run_verification_tests()

## Results

We run a small debug case first, then a larger case, and compare multiple seeds to show stochastic variability.

In [ ]:
# Debug case: 10x10 with 2 walkers/species
steps_test_1=1000
_, _, h_dbg, s_dbg = run_simulation(
    n=10, walkers_each_species=6, steps=steps_test_1, seed=np.random.seed(), snapshot_times=[0, int(steps_test_1 / 4), int(steps_test_1 / 2), steps_test_1]
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
for ax, t in zip(axes, [0, int(steps_test_1 / 4), int(steps_test_1 / 2), steps_test_1]):
    plot_grid(s_dbg[t], ax=ax, title=f"Debug t={t}")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
plot_species_history(h_dbg, ax=ax, title="Debug species fractions")
plt.show()

In [ ]:
# Larger run
_, _, h_main, s_main = run_simulation(
    n=50, walkers_each_species=10, steps=800, seed=2026, snapshot_times=[0, 100, 300, 800]
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
for ax, t in zip(axes, [0, 100, 300, 800]):
    plot_grid(s_main[t], ax=ax, title=f"Main t={t}")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.2))
plot_species_history(h_main, ax=ax, title="Main run species fractions")
plt.show()

In [ ]:
def summarize_outcome(history, dominance_threshold=0.90):
    final_species = history[-1, 1:4]
    winner_state = int(np.argmax(final_species)) + 1

    dom_time = None
    for t in range(history.shape[0]):
        if np.max(history[t, 1:4]) >= dominance_threshold:
            dom_time = t
            break

    coexist = np.max(history[-1, 1:4]) < dominance_threshold
    return winner_state, dom_time, coexist

seeds = [3, 7, 11, 19, 23, 29]
histories = {}
summary = []

for sd in seeds:
    _, _, h, _ = run_simulation(n=50, walkers_each_species=10, steps=800, seed=sd, snapshot_times=[0, 800])
    histories[sd] = h
    w, td, co = summarize_outcome(h)
    summary.append((sd, w, td, co))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True, constrained_layout=True)
for ax, sp in zip(axes, [ROCK, PAPER, SCISSORS]):
    for sd in seeds:
        ax.plot(histories[sd][:, sp], lw=1.6, alpha=0.8)
    ax.set_title(f"{STATE_NAMES[sp]} fraction")
    ax.set_xlabel("Time step")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Fraction")
plt.show()

print("seed | winner | time_to_dominance>=0.90 | coexist_end")
for sd, w, td, co in summary:
    print(f"{sd:4d} | {STATE_NAMES[w]:8s} | {str(td):>24} | {co}")

## Stretch Investigation: Parameter Sweep

We sweep initial walker count per species and compare win/coexistence fractions over repeated seeds.

In [ ]:
def run_batch(n=50, walkers_each_species=6, steps=600, seeds=range(12), dominance_threshold=0.90):
    winners = []
    coexist = []

    for sd in seeds:
        _, _, h, _ = run_simulation(n=n, walkers_each_species=walkers_each_species, steps=steps, seed=int(sd))
        w, _, co = summarize_outcome(h, dominance_threshold=dominance_threshold)
        winners.append(w)
        coexist.append(co)

    winners = np.array(winners)
    coexist = np.array(coexist, dtype=bool)

    return {
        "rock_win_frac": np.mean(winners == ROCK),
        "paper_win_frac": np.mean(winners == PAPER),
        "scissors_win_frac": np.mean(winners == SCISSORS),
        "coexist_frac": np.mean(coexist)
    }

k_values = [2, 6, 12, 20]
res = []
for k in k_values:
    out = run_batch(n=50, walkers_each_species=k, steps=600, seeds=range(12))
    out["k"] = k
    res.append(out)

x = np.array([d["k"] for d in res])
r = np.array([d["rock_win_frac"] for d in res])
p = np.array([d["paper_win_frac"] for d in res])
s = np.array([d["scissors_win_frac"] for d in res])
c = np.array([d["coexist_frac"] for d in res])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, r, "-o", label="Rock wins", color=STATE_COLORS[ROCK])
ax.plot(x, p, "-o", label="Paper wins", color=STATE_COLORS[PAPER])
ax.plot(x, s, "-o", label="Scissors wins", color=STATE_COLORS[SCISSORS])
ax.plot(x, c, "--s", label="Coexistence", color="black")
ax.set_xlabel("Initial walkers per species")
ax.set_ylabel("Fraction across runs")
ax.set_ylim(0, 1)
ax.set_title("Parameter sweep on initial seeding density")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Discussion and Reflection

Observed behavior is strongly stochastic: with identical parameters, different seeds can lead to different dominant species or persistent coexistence.

The model is intentionally minimal but captures key ideas from computational physics:
- local rules leading to emergent spatial patterns
- sensitivity to random fluctuations
- value of repeated trials and summary statistics

## Conclusion

This notebook built and validated a modular 2D Rock-Paper-Scissors territorial walker simulation, then analyzed outcomes across multiple random realizations and one parameter sweep.

In [ ]:
# Run-until-takeover + inline HTML animation of the whole trial
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display


def run_until_takeover_and_html(
    n=50,
    walkers_each_species=10,
    seed=2027,
    max_steps=5000,
    periodic=True,
    fps=20,
    interval_ms=50,
    frame_skip=1
):
    """
    Run simulation until complete takeover (all cells same non-empty species),
    then render an inline HTML animation in the notebook.

    frame_skip controls how often frames are stored.
    """
    grid, walkers, rng = initialize_grid_and_walkers(
        n=n, walkers_each_species=walkers_each_species, seed=seed
    )

    frames = [grid.copy()]
    takeover_step = None
    winner_species = None

    for t in range(1, max_steps + 1):
        step_system(grid, walkers, rng, periodic=periodic)

        if (t % frame_skip) == 0:
            frames.append(grid.copy())

        unique_states = np.unique(grid)
        if len(unique_states) == 1 and unique_states[0] in (ROCK, PAPER, SCISSORS):
            takeover_step = t
            winner_species = int(unique_states[0])
            if (t % frame_skip) != 0:
                frames.append(grid.copy())
            break

    if takeover_step is None:
        print(f"No complete takeover within {max_steps} steps.")
    else:
        print(
            f"Complete takeover at step {takeover_step} by {STATE_NAMES[winner_species]}."
        )

    fig, (ax, ax_key) = plt.subplots(
        1, 2, figsize=(8, 6), gridspec_kw={"width_ratios": [1.0, 0.42]}
    )
    im = ax.imshow(frames[0], cmap=cmap, norm=norm, origin="lower", interpolation="nearest")
    ax.set_xlabel("y")
    ax.set_ylabel("x")

    ax_key.axis("off")
    ax_key.set_title("Color key", fontsize=11)

    key_states = [ROCK, PAPER, SCISSORS, EMPTY]
    y0, dy = 0.82, 0.18
    for i, s in enumerate(key_states):
        y = y0 - i * dy
        ax_key.add_patch(
            plt.Rectangle(
                (0.08, y - 0.045),
                0.22,
                0.09,
                facecolor=STATE_COLORS[s],
                edgecolor="black",
                transform=ax_key.transAxes,
                clip_on=False,
            )
        )
        ax_key.text(
            0.36,
            y,
            f"{STATE_NAMES[s]} ({s})",
            transform=ax_key.transAxes,
            va="center",
            fontsize=10,
        )

    def _title_for_frame(i):
        sim_step = i * frame_skip
        if takeover_step is None:
            return f"RPS Trial (step {sim_step})"
        return f"RPS Trial (step {sim_step}) | takeover: {STATE_NAMES[winner_species]} at {takeover_step}"

    ax.set_title(_title_for_frame(0))

    def update(i):
        im.set_data(frames[i])
        ax.set_title(_title_for_frame(i))
        return [im]

    ani = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval_ms,
        blit=False
    )

    fig.tight_layout()
    html_anim = ani.to_jshtml(fps=fps)
    plt.close(fig)
    display(HTML(html_anim))
    print("Rendered inline HTML animation.")

    return {
        "takeover_step": takeover_step,
        "winner_species": winner_species,
        "num_frames": len(frames),
        "rendered_inline_html": True
    }

# Example run (uncomment to execute):
# result_takeover = run_until_takeover_and_html(
#     n=10,
#     walkers_each_species=5,
#     seed=24,
#     max_steps=5000,
#     periodic=True,
#     fps=20,
#     interval_ms=50,
#     frame_skip=2
# )
# result_takeover

print("Function 'run_until_takeover_and_html' is ready to use. Call it with your desired parameters.")

In [ ]:
result_takeover = run_until_takeover_and_html(
    n=50,
    walkers_each_species=300,
    seed=343,
    max_steps=1000,
    periodic=False,
    fps=40,
    interval_ms=50,
    frame_skip=2
)
result_takeover

In [ ]:
# Three-zone Y-symmetry (120° between boundaries)
def initialize_three_zones_and_run(
    n=40,
    seed=2027,
    max_steps=5000,
    periodic=False,
    fps=20,
    interval_ms=50,
    frame_skip=1
):
    """
    Initialize an n x n grid with three angular zones that meet at one center point
    (Y-shape, 120° symmetry).

    Territories are fully populated by species state at t=0, but walkers spawn ONLY on
    inter-zone borders. Walkers also cannot move into cells occupied by another walker.
    """
    rng = np.random.default_rng(seed)
    grid = np.zeros((n, n), dtype=np.int8)

    # Center point where the three sectors meet
    cx = (n - 1) / 2.0
    cy = (n - 1) / 2.0

    # Three unit direction vectors separated by 120 degrees
    # (horizontal, vertical) convention in plotting coordinates
    d_rock = np.array([0.0, 1.0])                      # up
    d_scissors = np.array([np.sqrt(3) / 2.0, -0.5])    # down-right
    d_paper = np.array([-np.sqrt(3) / 2.0, -0.5])      # down-left
    dirs = np.vstack([d_rock, d_paper, d_scissors])
    species_map = np.array([ROCK, PAPER, SCISSORS], dtype=np.int8)

    # Build full territory map (entire board populated by zones)
    for x in range(n):
        for y in range(n):
            vx = (y + 0.5) - cy   # horizontal component
            vy = (x + 0.5) - cx   # vertical component
            v = np.array([vx, vy])
            idx = int(np.argmax(dirs @ v))
            grid[x, y] = int(species_map[idx])

    # Spawn walkers only on inter-zone borders
    border_mask = np.zeros((n, n), dtype=bool)
    for x in range(n):
        for y in range(n):
            s = grid[x, y]
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                xn, yn = x + dx, y + dy
                if periodic:
                    xn %= n
                    yn %= n
                elif not (0 <= xn < n and 0 <= yn < n):
                    continue

                if grid[xn, yn] != s:
                    border_mask[x, y] = True
                    break

    walkers = []
    occupancy = -np.ones((n, n), dtype=np.int32)  # -1 means no walker in cell
    for x in range(n):
        for y in range(n):
            if border_mask[x, y]:
                walkers.append({"x": x, "y": y, "species": int(grid[x, y])})
                occupancy[x, y] = len(walkers) - 1

    frames = [grid.copy()]
    takeover_step = None
    winner_species = None

    moves = np.array([(-1, 0), (1, 0), (0, -1), (0, 1)], dtype=np.int8)

    for t in range(1, max_steps + 1):
        # Local species counts from walker distribution (for weighted capture probability)
        cell_counts = local_species_counts(walkers, n, n)

        for i in rng.permutation(len(walkers)):
            w = walkers[i]
            x_old, y_old = w["x"], w["y"]
            species = int(w["species"])

            # Try random directions until finding a free destination; otherwise stay put
            chosen = None
            for k in rng.permutation(4):
                dx, dy = moves[k]
                x_new, y_new = x_old + int(dx), y_old + int(dy)
                if periodic:
                    x_new %= n
                    y_new %= n
                else:
                    if not (0 <= x_new < n and 0 <= y_new < n):
                        continue

                if occupancy[x_new, y_new] == -1:
                    chosen = (x_new, y_new)
                    break

            if chosen is None:
                continue

            # Remove from old occupancy/count before conflict evaluation
            occupancy[x_old, y_old] = -1
            cell_counts[x_old, y_old, species] = max(0, cell_counts[x_old, y_old, species] - 1)

            x_new, y_new = chosen
            destination = int(grid[x_new, y_new])
            final_species = species

            if destination == EMPTY:
                grid[x_new, y_new] = species
            elif destination == species:
                pass
            else:
                normal_winner = winner(species, destination)
                attacker_count = int(cell_counts[x_new, y_new, species]) + 1
                defender_count = int(cell_counts[x_new, y_new, destination])

                if normal_winner == species:
                    conflict_winner = species
                else:
                    p_upset = against_flow_capture_probability(attacker_count, defender_count)
                    conflict_winner = species if rng.random() < p_upset else destination

                grid[x_old, y_old] = conflict_winner
                grid[x_new, y_new] = conflict_winner
                final_species = int(conflict_winner)

            w["x"], w["y"], w["species"] = x_new, y_new, int(final_species)
            occupancy[x_new, y_new] = i
            cell_counts[x_new, y_new, int(final_species)] += 1

        if (t % frame_skip) == 0:
            frames.append(grid.copy())

        unique_states = np.unique(grid)
        if len(unique_states) == 1 and unique_states[0] in (ROCK, PAPER, SCISSORS):
            takeover_step = t
            winner_species = int(unique_states[0])
            if (t % frame_skip) != 0:
                frames.append(grid.copy())
            break

    if takeover_step is None:
        print(f"No complete takeover within {max_steps} steps.")
    else:
        print(
            f"Complete takeover at step {takeover_step} by {STATE_NAMES[winner_species]}."
        )

    fig, (ax, ax_key) = plt.subplots(
        1, 2, figsize=(8, 6), gridspec_kw={"width_ratios": [1.0, 0.42]}
    )
    im = ax.imshow(frames[0], cmap=cmap, norm=norm, origin="lower", interpolation="nearest")
    ax.set_xlabel("y")
    ax.set_ylabel("x")

    ax_key.axis("off")
    ax_key.set_title("Color key", fontsize=11)

    key_states = [ROCK, PAPER, SCISSORS, EMPTY]
    y0, dy = 0.82, 0.18
    for i, s in enumerate(key_states):
        y = y0 - i * dy
        ax_key.add_patch(
            plt.Rectangle(
                (0.08, y - 0.045),
                0.22,
                0.09,
                facecolor=STATE_COLORS[s],
                edgecolor="black",
                transform=ax_key.transAxes,
                clip_on=False,
            )
        )
        ax_key.text(
            0.36,
            y,
            f"{STATE_NAMES[s]} ({s})",
            transform=ax_key.transAxes,
            va="center",
            fontsize=10,
        )

    def _title_for_frame(i):
        sim_step = i * frame_skip
        if takeover_step is None:
            return f"Three-Sector RPS Trial (step {sim_step})"
        return f"Three-Sector RPS Trial (step {sim_step}) | takeover: {STATE_NAMES[winner_species]} at {takeover_step}"

    ax.set_title(_title_for_frame(0))

    def update(i):
        im.set_data(frames[i])
        ax.set_title(_title_for_frame(i))
        return [im]

    ani = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval_ms,
        blit=False
    )

    from IPython.display import HTML, display

    fig.tight_layout()
    html_anim = ani.to_jshtml(fps=fps)
    plt.close(fig)
    display(HTML(html_anim))
    print("Rendered inline HTML animation.")

    return {
        "takeover_step": takeover_step,
        "winner_species": winner_species,
        "num_frames": len(frames),
        "rendered_inline_html": True
    }


# Run the three-zone simulation on a fixed 40x40 box with hard boundaries
result_three_zones = initialize_three_zones_and_run(
    n=1000,
    seed=2319,
    max_steps=1000,
    periodic=False,
    fps=20,
    interval_ms=50,
    frame_skip=2
)

print("\nThree-zone clash simulation complete!")
print(result_three_zones)

In [ ]:
# Three vertical territories (split on y) with periodic borders
def initialize_vertical_thirds_and_run(
    n=40,
    seed=2027,
    max_steps=5000,
    periodic=True,
    fps=20,
    interval_ms=50,
    frame_skip=1,
    border_spawn_prob=0.35,
    interface_perturb=2,
    flow_win_prob=0.9
):
    """
    Initialize an n x n grid using vertical thirds by y (column index), not x.
    Interfaces are slightly perturbed per row so they are not perfectly flat.

    Territories:
    - Left third: Paper
    - Middle third: Rock
    - Right third: Scissors

    Walkers spawn only on a randomized subset of border cells (sparse border population).
    Walkers may move into occupied cells; conflict is resolved at destination.

    flow_win_prob < 1 makes with-flow outcomes less deterministic.
    """
    rng = np.random.default_rng(seed)
    grid = np.zeros((n, n), dtype=np.int8)

    # Row-dependent perturbed interfaces around y = n/3 and y = 2n/3
    # Keep them ordered so we preserve left/middle/right territories.
    b1_base = n / 3.0
    b2_base = 2.0 * n / 3.0
    shifts = rng.integers(-interface_perturb, interface_perturb + 1, size=n)

    b1_by_x = np.zeros(n, dtype=np.int32)
    b2_by_x = np.zeros(n, dtype=np.int32)
    for x in range(n):
        b1 = int(np.clip(np.round(b1_base + shifts[x]), 1, n - 2))
        b2 = int(np.clip(np.round(b2_base + shifts[x]), b1 + 1, n - 1))
        b1_by_x[x] = b1
        b2_by_x[x] = b2

    # Build full territory map using y-split (vertical thirds)
    for x in range(n):
        b1, b2 = b1_by_x[x], b2_by_x[x]
        for y in range(n):
            if y < b1:
                grid[x, y] = PAPER
            elif y < b2:
                grid[x, y] = ROCK
            else:
                grid[x, y] = SCISSORS

    # Detect inter-zone borders
    border_mask = np.zeros((n, n), dtype=bool)
    for x in range(n):
        for y in range(n):
            s = grid[x, y]
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                xn = (x + dx) % n if periodic else x + dx
                yn = (y + dy) % n if periodic else y + dy

                if not periodic and not (0 <= xn < n and 0 <= yn < n):
                    continue

                if grid[xn, yn] != s:
                    border_mask[x, y] = True
                    break

    # Sparse random spawn on border cells
    walkers = []
    for x in range(n):
        for y in range(n):
            if border_mask[x, y] and (rng.random() < border_spawn_prob):
                walkers.append({"x": x, "y": y, "species": int(grid[x, y])})

    # Ensure each species has at least one walker
    for sp in [ROCK, PAPER, SCISSORS]:
        if not any(w["species"] == sp for w in walkers):
            candidates = np.argwhere((grid == sp) & border_mask)
            if len(candidates) > 0:
                idx = rng.integers(0, len(candidates))
                x0, y0 = candidates[idx]
                walkers.append({"x": int(x0), "y": int(y0), "species": int(sp)})

    frames = [grid.copy()]
    takeover_step = None
    winner_species = None

    moves = np.array([(-1, 0), (1, 0), (0, -1), (0, 1)], dtype=np.int8)

    for t in range(1, max_steps + 1):
        # Local species counts from current walker distribution
        cell_counts = local_species_counts(walkers, n, n)

        for i in rng.permutation(len(walkers)):
            w = walkers[i]
            x_old, y_old = w["x"], w["y"]
            species = int(w["species"])

            # Single random move; occupied destination is allowed
            dx, dy = moves[rng.integers(0, 4)]
            x_new, y_new = x_old + int(dx), y_old + int(dy)

            if periodic:
                x_new %= n
                y_new %= n
            else:
                if not (0 <= x_new < n and 0 <= y_new < n):
                    # hard wall: no move
                    continue

            # Remove from old local count before evaluating destination conflict
            cell_counts[x_old, y_old, species] = max(0, cell_counts[x_old, y_old, species] - 1)

            destination = int(grid[x_new, y_new])
            final_species = species

            if destination == EMPTY:
                grid[x_new, y_new] = species

            elif destination == species:
                pass

            else:
                normal_winner = winner(species, destination)
                attacker_count = int(cell_counts[x_new, y_new, species]) + 1
                defender_count = int(cell_counts[x_new, y_new, destination])

                if normal_winner == species:
                    # Optionally less deterministic even with the flow
                    conflict_winner = species if rng.random() < flow_win_prob else destination
                else:
                    p_upset = against_flow_capture_probability(attacker_count, defender_count)
                    conflict_winner = species if rng.random() < p_upset else destination

                grid[x_old, y_old] = conflict_winner
                grid[x_new, y_new] = conflict_winner
                final_species = int(conflict_winner)

            w["x"], w["y"], w["species"] = x_new, y_new, int(final_species)
            cell_counts[x_new, y_new, int(final_species)] += 1

        if (t % frame_skip) == 0:
            frames.append(grid.copy())

        unique_states = np.unique(grid)
        if len(unique_states) == 1 and unique_states[0] in (ROCK, PAPER, SCISSORS):
            takeover_step = t
            winner_species = int(unique_states[0])
            if (t % frame_skip) != 0:
                frames.append(grid.copy())
            break

    if takeover_step is None:
        print(f"No complete takeover within {max_steps} steps.")
    else:
        print(
            f"Complete takeover at step {takeover_step} by {STATE_NAMES[winner_species]}."
        )

    # Build a two-panel figure so the color key is always visible in the animation.
    fig, (ax, ax_key) = plt.subplots(
        1, 2, figsize=(8, 6), gridspec_kw={"width_ratios": [1.0, 0.42]}
    )
    im = ax.imshow(frames[0], cmap=cmap, norm=norm, origin="lower", interpolation="nearest")
    ax.set_xlabel("y")
    ax.set_ylabel("x")

    ax_key.axis("off")
    ax_key.set_title("Color key", fontsize=11)

    key_states = [ROCK, PAPER, SCISSORS, EMPTY]
    y0, dy = 0.82, 0.18
    for i, s in enumerate(key_states):
        y = y0 - i * dy
        ax_key.add_patch(
            plt.Rectangle(
                (0.08, y - 0.045),
                0.22,
                0.09,
                facecolor=STATE_COLORS[s],
                edgecolor="black",
                transform=ax_key.transAxes,
                clip_on=False,
            )
        )
        ax_key.text(
            0.36,
            y,
            f"{STATE_NAMES[s]} ({s})",
            transform=ax_key.transAxes,
            va="center",
            fontsize=10,
        )

    def _title_for_frame(i):
        sim_step = i * frame_skip
        if takeover_step is None:
            return f"Vertical Thirds (y-split) RPS Trial (step {sim_step})"
        return f"Vertical Thirds (y-split) RPS Trial (step {sim_step}) | takeover: {STATE_NAMES[winner_species]} at {takeover_step}"

    ax.set_title(_title_for_frame(0))

    def update(i):
        im.set_data(frames[i])
        ax.set_title(_title_for_frame(i))
        return [im]

    ani = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval_ms,
        blit=False
    )

    from IPython.display import HTML, display

    fig.tight_layout()
    html_anim = ani.to_jshtml(fps=fps)
    plt.close(fig)
    display(HTML(html_anim))
    print("Rendered inline HTML animation.")

    return {
        "takeover_step": takeover_step,
        "winner_species": winner_species,
        "num_frames": len(frames),
        "rendered_inline_html": True,
        "num_walkers": len(walkers)
    }


# Run the vertical-thirds y-split simulation with periodic boundaries
result_vertical_thirds = initialize_vertical_thirds_and_run(
    n=800,
    seed=986,
    max_steps=10000,
    periodic=True,
    fps=20,
    interval_ms=50,
    frame_skip=200,
    border_spawn_prob=0.35,
    interface_perturb=2,
    flow_win_prob=0.9
)

print("\nVertical thirds simulation complete!")
print(result_vertical_thirds)
